# Real-data CRNN adaptation v1

Start from the V4 continued checkpoint. Fine-tune on real print and cursive with synthetic replay. The completed 24-image CPU check reached 24/24 training words; this is not a generalization result.

Use Colab **T4 GPU**, then run cells in order. Package source data is external work; see the local source inventory. Filename-derived page splits do not establish independent writers/documents. The 475-image reserved test is never evaluated here. Run output saves to this package's Drive folder. Re-running training requires a new RUN_NAME; automatic resume is not implemented.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, shutil, zipfile, subprocess, sys, os
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0), torch.__version__)
PACKAGE = Path('/content/drive/MyDrive/4 Senior/Thesis/Manchu-CRNN-Real-Adaptation-v1')
assert (PACKAGE/'payload.zip').is_file(), f'Update PACKAGE to its actual Drive path: {PACKAGE}'


In [ ]:
# Stage locally; keep training reads off Drive. Source files/checkpoints stay untouched.
manifest = json.loads((PACKAGE/'MANIFEST.json').read_text())
assert manifest['sha256'] == '5a74b64c699d299272da8e65124a1ab7214d13f5be6e46925d9bb8d78986fec5'
payload = Path('/content/real_adaptation_v1_payload.zip')
shutil.copy2(PACKAGE/'payload.zip', payload)
assert hashlib.sha256(payload.read_bytes()).hexdigest() == manifest['sha256']
with zipfile.ZipFile(payload) as archive:
    archive.extractall('/content/manchu_real_adaptation_v1')
ROOT = Path('/content/manchu_real_adaptation_v1/research')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'Pillow', 'numpy'], check=True)
protocol = json.loads((ROOT/'data/real_adaptation_v1/PROTOCOL.json').read_text())
print(protocol['counts'])
print(protocol['limits'])
assert json.loads((ROOT/'research_runs/real_adaptation_v1_pilot/RESULT.json').read_text())['pilot_gate_passed']


## Full adaptation

Batch 24 = eight cursive + eight printed + eight synthetic images. Adam 1e-4, clip norm 5, 500 updates per reporting epoch, at most ten epochs, patience three. An epoch is an update budget, not a full dataset pass. Select lowest mean printed/cursive development CER; report synthetic retention separately. Training starts from the original V4 continued weights, not the overfit pilot. Final test stays reserved. CUDA preprocessing and model operations run on GPU; Parquet/Pillow decoding stays on CPU.


In [ ]:
RUN_TRAINING = True
RUN_NAME = 'real_adaptation_v1_cuda_01'
if RUN_TRAINING:
    env = dict(os.environ, MANCHU_ADAPT_RUN_ROOT=str(PACKAGE/'runs'))
    subprocess.run([sys.executable, str(ROOT/'experiments/adapt_real_v1.py'), 'train',
                    '--device', 'cuda', '--name', RUN_NAME], cwd=ROOT, env=env, check=True)
else:
    print('Training disabled')


In [ ]:
history = json.loads((PACKAGE/'runs'/RUN_NAME/'history.json').read_text())
for epoch in history:
    print(epoch['step'], {k: {'accuracy': round(v['accuracy'],4), 'cer': round(v['cer'],4)}
                          for k,v in epoch['metrics'].items()})
print((PACKAGE/'runs'/RUN_NAME/'RESULT.json').read_text())
